# Tuần 16: Hồi Quy (Regression)
### Bước đầu tiên vào Machine Learning!

---

## Hồi quy là gì?

**Hồi quy** là kỹ thuật giúp ta **dự đoán một con số** dựa trên thông tin đã biết.

| Thông tin đã biết | → Dự đoán |
|---|---|
| Học **10 tiếng** | → Điểm thi khoảng **bao nhiêu**? |
| Chi **50 triệu** quảng cáo | → Doanh số **bao nhiêu**? |
| Nhà **80m², 3 phòng ngủ** | → Giá **bao nhiêu**? |

> Giống như bạn thấy bạn mình học 5 tiếng được 50 điểm, học 8 tiếng được 80 điểm → bạn "đoán" học 10 tiếng sẽ khoảng 100 điểm. **Đó chính là hồi quy!**

In [ ]:
# Cài đặt thư viện cần dùng (chạy cell này trước)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder

# Cấu hình biểu đồ đẹp hơn
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Sẵn sàng học bài!')

---
# Phần 1: Hồi quy tuyến tính đơn (Simple Linear Regression)

## Ý tưởng cốt lõi

Hãy tưởng tượng bạn vẽ các điểm trên giấy:
- **Trục ngang (X)** = số giờ học
- **Trục dọc (Y)** = điểm thi

Hồi quy tuyến tính = **vẽ một đường thẳng** đi qua các điểm sao cho gần nhất với tất cả.

### Công thức:
## $$Y = a \times X + b$$

| Ký hiệu | Tên | Ý nghĩa |
|---|---|---|
| **Y** | Giá trị dự đoán | Điểm thi |
| **X** | Dữ liệu đầu vào | Số giờ học |
| **a** | Hệ số góc (slope) | X tăng 1 → Y tăng bao nhiêu |
| **b** | Hệ số chặn (intercept) | Giá trị Y khi X = 0 |

Hãy xem biểu đồ bên dưới để hiểu trực quan!

In [ ]:
# === VÍ DỤ 1: Dự đoán ĐIỂM THI từ SỐ GIỜ HỌC ===

# Bước 1: Tạo dữ liệu mẫu (100 sinh viên)
np.random.seed(42)  # Để kết quả giống nhau mỗi lần chạy
hours = np.random.randint(1, 20, 100)                    # Số giờ học: ngẫu nhiên từ 1-19
grades = 5 * hours + np.random.normal(0, 5, 100)         # Điểm = 5 × giờ + một chút ngẫu nhiên

# Xem thử 5 sinh viên đầu tiên
print('Số giờ học  :', hours[:5])
print('Điểm thi   :', np.round(grades[:5], 1))
print(f'\nTổng cộng có {len(hours)} sinh viên')

In [ ]:
# Bước 2: Vẽ biểu đồ scatter - xem mối quan hệ giờ học vs điểm thi

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ bên trái: Chỉ các điểm dữ liệu
axes[0].scatter(hours, grades, alpha=0.5, color='steelblue', edgecolors='navy')
axes[0].set_xlabel('Số giờ học', fontsize=13)
axes[0].set_ylabel('Điểm thi', fontsize=13)
axes[0].set_title('Dữ liệu thô: Giờ học vs Điểm', fontsize=14)

# Biểu đồ bên phải: Các điểm + đường hồi quy
axes[1].scatter(hours, grades, alpha=0.5, color='steelblue', edgecolors='navy', label='Dữ liệu thực')
# Vẽ đường hồi quy (tạm tính nhanh)
x_line = np.linspace(0, 20, 100)
y_line = 5 * x_line  # Đường lý tưởng Y = 5X
axes[1].plot(x_line, y_line, color='red', linewidth=2, label='Đường hồi quy (Y = 5X)')
axes[1].set_xlabel('Số giờ học', fontsize=13)
axes[1].set_ylabel('Điểm thi', fontsize=13)
axes[1].set_title('Đường hồi quy: "đường thẳng tốt nhất"', fontsize=14)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print('Nhìn biểu đồ bên phải: đường đỏ đi qua "giữa" các điểm xanh')
print('→ Đó chính là đường hồi quy!')

## Máy tính "học" như thế nào? — Gradient Descent

Máy tính tìm đường thẳng tốt nhất bằng một kỹ thuật gọi là **Gradient Descent** (Hạ dốc theo độ dốc).

### Hình dung bằng đời thường:

Hãy tưởng tượng bạn đang **đứng trên đỉnh một ngọn đồi trong sương mù** — bạn không thấy được đâu là chỗ thấp nhất, nhưng bạn **cảm nhận được chân mình đang đứng dốc hướng nào**.

Bạn làm gì? → **Bước từng bước nhỏ theo hướng dốc xuống!**

Cứ thế, từng bước, từng bước... cho đến khi bạn tới **chỗ bằng phẳng nhất** = đáy thung lũng = **sai số nhỏ nhất!**

### Quy trình 4 bước:

| Bước | Hành động | Ví dụ trên đồi |
|------|-----------|-----------------|
| 1 | Bắt đầu với a, b **ngẫu nhiên** | Đứng ở vị trí bất kỳ trên đồi |
| 2 | Tính **sai số** (MSE) giữa dự đoán và thực tế | Xem mình đang ở **cao** hay **thấp** |
| 3 | Tính **độ dốc** (gradient) → biết nên tăng/giảm a, b | Cảm nhận chân đang **dốc hướng nào** |
| 4 | **Cập nhật** a, b theo hướng giảm sai số | Bước một bước **xuống dốc** |

Lặp lại bước 2-3-4 cho đến khi sai số không giảm nữa → **tìm được a, b tốt nhất!**

### Learning Rate — Bước chân to hay nhỏ?

- **Learning rate LỚN** = bước chân TO → đi nhanh nhưng có thể **nhảy qua** đáy
- **Learning rate NHỎ** = bước chân NHỎ → đi chậm nhưng **chính xác** hơn
- Cần chọn learning rate **vừa phải** — không quá to, không quá nhỏ

Chạy cell bên dưới để xem Gradient Descent hoạt động trực quan!

In [ ]:
# Minh họa: Sai số (error) là khoảng cách từ điểm thực đến đường dự đoán

fig, ax = plt.subplots(figsize=(10, 6))

# Lấy 15 điểm đầu để dễ nhìn
n_show = 15
h = hours[:n_show]
g = grades[:n_show]
g_pred = 5 * h  # Dự đoán

# Vẽ các điểm thực
ax.scatter(h, g, color='steelblue', s=80, zorder=5, label='Điểm thực tế')

# Vẽ đường hồi quy
x_line = np.linspace(0, 20, 100)
ax.plot(x_line, 5 * x_line, color='red', linewidth=2, label='Đường dự đoán')

# Vẽ các đoạn sai số (đường đứt nét từ điểm thực → đường dự đoán)
for i in range(n_show):
    ax.plot([h[i], h[i]], [g[i], g_pred[i]], 'g--', linewidth=1.5, alpha=0.7)

ax.plot([], [], 'g--', label='Sai số (error)')  # Thêm legend
ax.set_xlabel('Số giờ học', fontsize=13)
ax.set_ylabel('Điểm thi', fontsize=13)
ax.set_title('Sai số = khoảng cách từ điểm thực → đường dự đoán', fontsize=14)
ax.legend(fontsize=12)
plt.show()

print('Các đường xanh lá (đứt nét) = SAI SỐ')
print('Mục tiêu: tìm đường đỏ sao cho TỔNG các đường xanh lá NGẮN NHẤT')

In [ ]:
# === MINH HỌA GRADIENT DESCENT ===
# Xem máy tính tìm đường thẳng tốt nhất như thế nào

# Dữ liệu đơn giản: 5 sinh viên
X_demo = np.array([1, 2, 3, 4, 5])
Y_demo = np.array([6, 11, 14, 21, 26])

# --- Gradient Descent thủ công ---
a_gd = 0.0    # Bắt đầu với a = 0 (đoán bừa)
b_gd = 0.0    # Bắt đầu với b = 0 (đoán bừa)
lr = 0.01     # Learning rate = bước chân
n = len(X_demo)

# Lưu lại quá trình học
history_a = [a_gd]
history_b = [b_gd]
history_mse = []

for step in range(150):
    # Dự đoán với a, b hiện tại
    Y_pred_gd = a_gd * X_demo + b_gd
    
    # Tính sai số MSE
    mse_gd = np.mean((Y_demo - Y_pred_gd) ** 2)
    history_mse.append(mse_gd)
    
    # Tính gradient (độ dốc) → biết nên điều chỉnh a, b thế nào
    grad_a = -2/n * np.sum(X_demo * (Y_demo - Y_pred_gd))  # Độ dốc theo a
    grad_b = -2/n * np.sum(Y_demo - Y_pred_gd)              # Độ dốc theo b
    
    # Cập nhật a, b: bước 1 bước xuống dốc
    a_gd = a_gd - lr * grad_a
    b_gd = b_gd - lr * grad_b
    
    history_a.append(a_gd)
    history_b.append(b_gd)

# --- VẼ KẾT QUẢ ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Biểu đồ 1: Đường thẳng thay đổi theo từng bước
colors = plt.cm.Reds(np.linspace(0.2, 1, 6))
for i, step in enumerate([0, 5, 15, 40, 80, 149]):
    a_s, b_s = history_a[step], history_b[step]
    axes[0].plot(X_demo, a_s * X_demo + b_s, color=colors[i], alpha=0.7,
                 label=f'Bước {step}: a={a_s:.1f}, b={b_s:.1f}')
axes[0].scatter(X_demo, Y_demo, color='steelblue', s=100, zorder=5, label='Dữ liệu thực')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].set_title('Đường thẳng "tiến hóa" qua từng bước')
axes[0].legend(fontsize=9)

# Biểu đồ 2: MSE giảm dần (đi xuống đồi)
axes[1].plot(history_mse, color='red', linewidth=2)
axes[1].set_xlabel('Số bước (iterations)')
axes[1].set_ylabel('Sai số MSE')
axes[1].set_title('Sai số GIẢM DẦN = Đang đi xuống đồi!')
axes[1].axhline(y=min(history_mse), color='green', linestyle='--', alpha=0.5, label=f'MSE thấp nhất = {min(history_mse):.2f}')
axes[1].legend()

# Biểu đồ 3: So sánh learning rate
axes[2].set_title('Learning Rate: to vs nhỏ')
for lr_test, color, label in [(0.001, 'blue', 'LR=0.001 (chậm)'), 
                                (0.01, 'green', 'LR=0.01 (vừa)'), 
                                (0.05, 'red', 'LR=0.05 (nhanh)')]:
    a_t, b_t = 0.0, 0.0
    mse_hist = []
    for _ in range(80):
        pred = a_t * X_demo + b_t
        mse_hist.append(np.mean((Y_demo - pred) ** 2))
        g_a = -2/n * np.sum(X_demo * (Y_demo - pred))
        g_b = -2/n * np.sum(Y_demo - pred)
        a_t -= lr_test * g_a
        b_t -= lr_test * g_b
    axes[2].plot(mse_hist, color=color, linewidth=2, label=label)
axes[2].set_xlabel('Số bước')
axes[2].set_ylabel('Sai số MSE')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f'Kết quả Gradient Descent: a = {a_gd:.2f}, b = {b_gd:.2f}')
print(f'→ Công thức: Y = {a_gd:.2f} × X + {b_gd:.2f}')
print(f'→ MSE cuối cùng = {history_mse[-1]:.4f}')
print()
print('NHÌN BIỂU ĐỒ:')
print('  Trái : Đường thẳng từ "sai be bét" → dần dần "khớp" với dữ liệu')
print('  Giữa : Sai số giảm dần giống đi xuống đồi')
print('  Phải : Learning rate nhỏ = chậm, to = nhanh nhưng rung lắc')

---
## Thực hành: Huấn luyện mô hình hồi quy

Quy trình 4 bước:

| Bước | Code | Ý nghĩa |
|------|------|----------|
| 1 | `train_test_split()` | Chia dữ liệu: 80% học, 20% kiểm tra |
| 2 | `model.fit()` | "Dạy" mô hình bằng dữ liệu training |
| 3 | `model.predict()` | Dự đoán trên dữ liệu test |
| 4 | `r2_score()`, `MSE` | Đánh giá: mô hình tốt hay dở? |

In [ ]:
# === BƯỚC 1: CHIA DỮ LIỆU ===

hours_train = hours[:80]     # 80 sinh viên đầu → để HỌC
grades_train = grades[:80]

hours_test = hours[80:]      # 20 sinh viên cuối → để KIỂM TRA
grades_test = grades[80:]

print(f'Dữ liệu huấn luyện: {len(hours_train)} sinh viên')
print(f'Dữ liệu kiểm tra  : {len(hours_test)} sinh viên')
print()
print('Tại sao phải chia?')
print('→ Giống bạn HỌC BÀI (training) rồi làm BÀI KIỂM TRA (test) bằng đề MỚI')
print('→ Để xem bạn thực sự hiểu bài hay chỉ HỌC VẸT!')

In [ ]:
# === BƯỚC 2: HUẤN LUYỆN MÔ HÌNH ===

model = LinearRegression()                                    # Tạo mô hình
model.fit(hours_train.reshape(-1, 1), grades_train)           # Dạy mô hình
#         ^^^^^^^^^^^^^^^^^^^^^^^^                             
#         reshape(-1,1): sklearn yêu cầu dữ liệu dạng bảng (2 chiều)

a = model.coef_[0]       # Hệ số góc
b = model.intercept_     # Hệ số chặn

print(f'Mô hình đã học được:')
print(f'  a (hệ số góc)  = {a:.2f}')
print(f'  b (hệ số chặn) = {b:.2f}')
print(f'\n→ Công thức: Điểm = {a:.2f} × Giờ_học + ({b:.2f})')
print(f'\n→ Nghĩa là: Mỗi giờ học thêm → điểm tăng khoảng {a:.1f} điểm')

# --- Thử dự đoán ---
print(f'\nThử dự đoán điểm cho các trường hợp:')
print('=' * 40)
for gio in [5, 10, 15, 18]:
    diem_du_doan = a * gio + b
    print(f'  Học {gio:2d} giờ → Điểm dự đoán: {diem_du_doan:.1f}')

In [ ]:
# === BƯỚC 3 & 4: DỰ ĐOÁN VÀ ĐÁNH GIÁ ===

grades_pred = model.predict(hours_test.reshape(-1, 1))    # Dự đoán

r2 = r2_score(grades_test, grades_pred)                    # R²
mse = mean_squared_error(grades_test, grades_pred)         # MSE

# Vẽ biểu đồ so sánh
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Điểm thực vs điểm dự đoán
axes[0].scatter(hours_test, grades_test, color='steelblue', s=80, label='Điểm thực tế', zorder=5)
axes[0].scatter(hours_test, grades_pred, color='red', s=80, marker='x', label='Điểm dự đoán', zorder=5)
x_line = np.linspace(0, 20, 100)
axes[0].plot(x_line, a * x_line + b, color='red', alpha=0.3, linewidth=2)
axes[0].set_xlabel('Số giờ học')
axes[0].set_ylabel('Điểm thi')
axes[0].set_title('Điểm thực (xanh) vs Dự đoán (đỏ)')
axes[0].legend()

# Biểu đồ 2: Thanh so sánh
x_pos = np.arange(len(grades_test))
width = 0.35
axes[1].bar(x_pos - width/2, grades_test, width, label='Thực tế', color='steelblue', alpha=0.7)
axes[1].bar(x_pos + width/2, grades_pred, width, label='Dự đoán', color='salmon', alpha=0.7)
axes[1].set_xlabel('Sinh viên thứ #')
axes[1].set_ylabel('Điểm thi')
axes[1].set_title('So sánh từng sinh viên')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nĐÁNH GIÁ MÔ HÌNH:')
print(f'  R² = {r2:.4f} → Mô hình giải thích được {r2*100:.1f}% sự thay đổi của điểm')
print(f'  MSE = {mse:.2f} → Sai số trung bình')
print(f'\n  R² gần 1 = TỐT | R² gần 0 = TỆ')

### Hiểu R² bằng hình ảnh

| R² | Ý nghĩa | Hình dung |
|---|---|---|
| **0.95 - 1.0** | Xuất sắc | Các điểm nằm sát đường thẳng |
| **0.80 - 0.95** | Tốt | Các điểm nằm gần đường thẳng, lệch ít |
| **0.50 - 0.80** | Trung bình | Điểm rải rác, xu hướng vẫn thấy được |
| **< 0.50** | Kém | Điểm rải lung tung, đường thẳng không giúp gì |

---
# Phần 2: Dữ liệu thực tế — Doanh số vs Quảng cáo

Bây giờ ta dùng dữ liệu **thật** từ file `data_simple_reg.csv`:
- **X** = Chi phí quảng cáo
- **Y** = Doanh số bán hàng

Câu hỏi: **Chi thêm tiền quảng cáo có tăng doanh số không? Tăng bao nhiêu?**

In [ ]:
# Đọc dữ liệu
df = pd.read_csv('data_simple_reg.csv')
print(f'Dữ liệu có {len(df)} dòng')
print()
df.head(10)

In [ ]:
# Xử lý dữ liệu thiếu (NaN)
print(f'Trước khi xử lý: {len(df)} dòng')
print(f'Số ô bị thiếu dữ liệu:')
print(df.isnull().sum())

df.dropna(inplace=True)   # Xóa dòng có dữ liệu thiếu
print(f'\nSau khi xử lý: {len(df)} dòng')

In [ ]:
# Xem mối quan hệ Quảng cáo → Doanh số

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Scatter plot
axes[0].scatter(df['Quang_Cao'], df['Doanh_So'], alpha=0.3, s=10, color='steelblue')
axes[0].set_xlabel('Chi phí Quảng cáo')
axes[0].set_ylabel('Doanh số')
axes[0].set_title('Quảng cáo vs Doanh số')

# Phân bố Quảng cáo
axes[1].hist(df['Quang_Cao'], bins=30, color='steelblue', alpha=0.7, edgecolor='navy')
axes[1].set_xlabel('Chi phí Quảng cáo')
axes[1].set_title('Phân bố: Chi phí Quảng cáo')

# Phân bố Doanh số
axes[2].hist(df['Doanh_So'], bins=30, color='salmon', alpha=0.7, edgecolor='darkred')
axes[2].set_xlabel('Doanh số')
axes[2].set_title('Phân bố: Doanh số')

plt.tight_layout()
plt.show()

print('Nhìn biểu đồ bên trái: Quảng cáo tăng → Doanh số tăng rõ ràng!')
print('→ Rất phù hợp để dùng hồi quy tuyến tính')

In [ ]:
# Huấn luyện mô hình trên dữ liệu thực

X = df['Quang_Cao']
Y = df['Doanh_So']

# Chia dữ liệu (cách chuyên nghiệp: dùng train_test_split)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, 
    test_size=0.1,      # 10% để test
    random_state=42      # Seed để kết quả lặp lại được
)

print(f'Training: {len(X_train)} mẫu')
print(f'Testing : {len(X_test)} mẫu')

# Huấn luyện
model_ds = LinearRegression()
model_ds.fit(X_train.values.reshape(-1, 1), Y_train.values)

a = model_ds.coef_[0]
b = model_ds.intercept_

print(f'\nKết quả:')
print(f'  Doanh_So = {a:.2f} × Quang_Cao + {b:.2f}')
print(f'\n  → Mỗi triệu chi cho quảng cáo → doanh số tăng {a:.2f} triệu!')

In [ ]:
# Đánh giá và trực quan

Y_pred = model_ds.predict(X_test.values.reshape(-1, 1))
r2 = r2_score(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Dữ liệu + đường hồi quy
axes[0].scatter(X_test, Y_test, alpha=0.5, color='steelblue', label='Thực tế')
x_line = np.linspace(X.min(), X.max(), 100)
axes[0].plot(x_line, a * x_line + b, color='red', linewidth=2, label='Đường hồi quy')
axes[0].set_xlabel('Chi phí Quảng cáo')
axes[0].set_ylabel('Doanh số')
axes[0].set_title(f'Hồi quy: Quảng cáo → Doanh số (R² = {r2:.4f})')
axes[0].legend()

# Biểu đồ 2: Thực tế vs Dự đoán (scatter)
axes[1].scatter(Y_test, Y_pred, alpha=0.3, color='steelblue')
min_val = min(Y_test.min(), Y_pred.min())
max_val = max(Y_test.max(), Y_pred.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Đường hoàn hảo')
axes[1].set_xlabel('Doanh số THỰC TẾ')
axes[1].set_ylabel('Doanh số DỰ ĐOÁN')
axes[1].set_title('Thực tế vs Dự đoán (càng sát đường đỏ = càng tốt)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'R² = {r2:.4f} → Mô hình giải thích {r2*100:.1f}% — RẤT TỐT!')
print(f'MSE = {mse:.2f}')

---
# Phần 3: Hồi quy tuyến tính bội (Multiple Regression)

## Khi doanh số phụ thuộc NHIỀU yếu tố

Trong thực tế, doanh số không chỉ phụ thuộc quảng cáo mà còn **PR**, **Online**, **KOL**...

### Công thức:
## $$Y = a_1 \times X_1 + a_2 \times X_2 + ... + b$$

### Vấn đề: Biến phân loại (chữ → số)

Cột `Quang_Cao` có giá trị: "High", "Low", "Medium-High", "Medium-Low"  
Máy tính không hiểu chữ! → Dùng **One-Hot Encoding** để chuyển:

| Quang_Cao | → High | Low | Medium-High | Medium-Low |
|---|---|---|---|---|
| High | 1 | 0 | 0 | 0 |
| Low | 0 | 1 | 0 | 0 |
| Medium-High | 0 | 0 | 1 | 0 |

> Giống bạn điền bảng khảo sát: thay vì viết "thích màu đỏ", bạn tích vào ô: ✅ Đỏ | ❌ Xanh | ❌ Vàng

In [ ]:
# Đọc dữ liệu mới
df2 = pd.read_csv('data_multiple_reg.csv')
print('Dữ liệu gốc:')
df2.head()

In [ ]:
# === SO SÁNH: Dùng 2 biến số (PR + Online) vs Thêm biến phân loại ===

Y2 = df2['Doanh_So']

# ---- Cách 1: Chỉ dùng PR + Online ----
X2_simple = df2[['PR', 'Online']]
X2s_train, X2s_test, Y2s_train, Y2s_test = train_test_split(X2_simple, Y2, test_size=0.2, random_state=42)

model_simple = LinearRegression()
model_simple.fit(X2s_train, Y2s_train)
r2_simple = r2_score(Y2s_test, model_simple.predict(X2s_test))

# ---- Cách 2: Thêm Quang_Cao (One-Hot Encoding) ----
qc_encoded = pd.get_dummies(df2['Quang_Cao'])   # Chuyển chữ → số
X2_full = pd.concat([qc_encoded, df2[['PR', 'Online']]], axis=1)

X2f_train, X2f_test, Y2f_train, Y2f_test = train_test_split(X2_full, Y2, test_size=0.2, random_state=42)

model_full = LinearRegression()
model_full.fit(X2f_train, Y2f_train)
r2_full = r2_score(Y2f_test, model_full.predict(X2f_test))

# ---- So sánh trực quan ----
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['Chỉ PR + Online', 'Thêm Quảng cáo\n(One-Hot Encoding)'], 
              [r2_simple, r2_full], 
              color=['#ff9999', '#66bb6a'], 
              edgecolor='black', width=0.5)

# Thêm số trên cột
for bar, val in zip(bars, [r2_simple, r2_full]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'R² = {val:.3f}', ha='center', fontsize=14, fontweight='bold')

ax.set_ylabel('R² (càng cao càng tốt)', fontsize=13)
ax.set_title('Thêm biến phân loại giúp mô hình TỐT HƠN RẤT NHIỀU!', fontsize=14)
ax.set_ylim(0, 1.1)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='R² = 1 (hoàn hảo)')
ax.legend()
plt.show()

print(f'Chỉ dùng PR + Online       → R² = {r2_simple:.3f} ({r2_simple*100:.1f}%)')
print(f'Thêm biến Quảng cáo (chữ)  → R² = {r2_full:.3f} ({r2_full*100:.1f}%)')
print(f'\n→ Cải thiện +{(r2_full - r2_simple)*100:.1f}% nhờ One-Hot Encoding!')

---
# Phần 4: Hồi quy Logistic — Phân loại Tốt / Xấu

## Khác gì hồi quy tuyến tính?

| | Hồi quy tuyến tính | Hồi quy Logistic |
|---|---|---|
| **Dự đoán** | Một CON SỐ (điểm, giá, doanh số) | Một NHÃN (Tốt/Xấu, Có/Không) |
| **Kết quả** | -∞ đến +∞ | 0 đến 1 (xác suất) |
| **Ví dụ** | Điểm thi = 75.5 | Xác suất đậu = 0.85 → **Đậu** |

### Hàm Sigmoid — "bộ ép" xác suất

Hồi quy logistic dùng hàm Sigmoid để ép kết quả về **0-1**:
- Kết quả > 0.5 → Dự đoán: **Tốt** (1)
- Kết quả ≤ 0.5 → Dự đoán: **Xấu** (0)

In [ ]:
# Vẽ hàm Sigmoid để hiểu trực quan

x_sig = np.linspace(-10, 10, 200)
y_sig = 1 / (1 + np.exp(-x_sig))   # Công thức Sigmoid

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_sig, y_sig, color='blue', linewidth=3)
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='Ngưỡng 0.5')
ax.fill_between(x_sig, y_sig, 0.5, where=(y_sig > 0.5), alpha=0.2, color='green', label='Vùng DỰ ĐOÁN = 1 (Tốt)')
ax.fill_between(x_sig, y_sig, 0.5, where=(y_sig <= 0.5), alpha=0.2, color='red', label='Vùng DỰ ĐOÁN = 0 (Xấu)')
ax.set_xlabel('Giá trị đầu vào', fontsize=13)
ax.set_ylabel('Xác suất (0 → 1)', fontsize=13)
ax.set_title('Hàm Sigmoid: ép mọi giá trị về khoảng 0-1', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.05)
plt.show()

print('Xác suất > 0.5 → Dự đoán: TỐT (1)')
print('Xác suất ≤ 0.5 → Dự đoán: XẤU (0)')

In [ ]:
# === THỰC HÀNH: Dự đoán đánh giá khách hàng (Tốt/Xấu) ===

df3 = pd.read_csv('data_logistic_reg.csv')
df3.dropna(inplace=True)

print(f'Dữ liệu: {len(df3)} đánh giá của khách hàng')
print(f'\nCác cột:')
print(f'  - Chat_Luong_Tai_Xe: chất lượng tài xế (thang điểm)')
print(f'  - Thoi_Gian_vs_Du_Kien: thời gian so với dự kiến')
print(f'  - Danh_Gia: {df3["Danh_Gia"].unique()} (cần dự đoán!)')
print(f'\nSố lượng mỗi loại:')
print(df3['Danh_Gia'].value_counts())

In [ ]:
# Chuyển nhãn chữ → số
encoder = LabelEncoder()
df3['Danh_Gia_So'] = encoder.fit_transform(df3['Danh_Gia'])  
print('Chuyển đổi:')
print(f'  {encoder.classes_[0]} → 0')
print(f'  {encoder.classes_[1]} → 1')

# Chuẩn bị dữ liệu
X3 = df3[['Chat_Luong_Tai_Xe', 'Thoi_Gian_vs_Du_Kien']]
Y3 = df3['Danh_Gia_So']

X3_train, X3_test, Y3_train, Y3_test = train_test_split(X3, Y3, test_size=0.1, random_state=42)

# Huấn luyện
model_log = LogisticRegression(max_iter=1000)
model_log.fit(X3_train, Y3_train)

# Dự đoán
Y3_pred = model_log.predict(X3_test)

print(f'\nHuấn luyện xong trên {len(X3_train)} mẫu!')

In [ ]:
# === ĐÁNH GIÁ MÔ HÌNH PHÂN LOẠI ===

acc = accuracy_score(Y3_test, Y3_pred)
prec = precision_score(Y3_test, Y3_pred)
rec = recall_score(Y3_test, Y3_pred)
f1 = f1_score(Y3_test, Y3_pred)

# Vẽ biểu đồ các chỉ số
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Các chỉ số đánh giá
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [acc, prec, rec, f1]
colors = ['#42a5f5', '#66bb6a', '#ffa726', '#ef5350']
bars = axes[0].bar(metrics, values, color=colors, edgecolor='black', width=0.6)
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{val:.3f}', ha='center', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Các chỉ số đánh giá mô hình phân loại', fontsize=14)
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)

# Biểu đồ 2: Confusion matrix đơn giản
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(Y3_test, Y3_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Dự đoán: Xấu', 'Dự đoán: Tốt'],
            yticklabels=['Thực tế: Xấu', 'Thực tế: Tốt'])
axes[1].set_title('Ma trận nhầm lẫn (Confusion Matrix)', fontsize=14)

plt.tight_layout()
plt.show()

print('GIẢI THÍCH CÁC CHỈ SỐ:')
print(f'  Accuracy  = {acc:.3f} → Tổng thể, mô hình đoán đúng {acc*100:.1f}%')
print(f'  Precision = {prec:.3f} → Khi mô hình nói "Tốt" → thực sự tốt {prec*100:.1f}%')
print(f'  Recall    = {rec:.3f} → Trong tất cả cái thực sự "Tốt" → tìm ra {rec*100:.1f}%')
print(f'  F1-Score  = {f1:.3f} → Trung bình hài hòa của Precision và Recall')

---
# Phần 5: Ví dụ bổ sung — Gần gũi đời thường

## Ví dụ: Dự đoán tiền điện hàng tháng

In [ ]:
# Ví dụ: Dự đoán tiền điện từ số giờ dùng điều hòa

gio_dieu_hoa = np.array([50, 80, 100, 120, 150, 180, 200])     # Giờ dùng
tien_dien = np.array([500, 750, 900, 1100, 1400, 1600, 1800])  # Nghìn đồng

# Huấn luyện
model_dien = LinearRegression()
model_dien.fit(gio_dieu_hoa.reshape(-1, 1), tien_dien)

a_dien = model_dien.coef_[0]
b_dien = model_dien.intercept_

# Vẽ
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(gio_dieu_hoa, tien_dien, color='orange', s=100, zorder=5, label='Dữ liệu thực')
x_line = np.linspace(30, 220, 100)
ax.plot(x_line, a_dien * x_line + b_dien, color='red', linewidth=2, label=f'Y = {a_dien:.0f}X + {b_dien:.0f}')

# Dự đoán cho 130 giờ
du_doan_130 = a_dien * 130 + b_dien
ax.scatter([130], [du_doan_130], color='green', s=150, marker='*', zorder=5, label=f'Dự đoán 130 giờ = {du_doan_130:.0f}K')
ax.axvline(x=130, color='green', linestyle=':', alpha=0.5)

ax.set_xlabel('Giờ dùng điều hòa', fontsize=13)
ax.set_ylabel('Tiền điện (nghìn đồng)', fontsize=13)
ax.set_title('Dự đoán tiền điện từ giờ dùng điều hòa', fontsize=14)
ax.legend(fontsize=12)
plt.show()

print(f'Công thức: Tiền = {a_dien:.0f} × Giờ + {b_dien:.0f}')
print(f'→ Mỗi giờ dùng thêm → tiền tăng khoảng {a_dien:.0f} nghìn')
print(f'→ Dùng 130 giờ → Tiền ≈ {du_doan_130:.0f} nghìn đồng')

---
# Tóm tắt — 7 điểm cần nhớ

| # | Kiến thức | Ghi nhớ |
|---|---|---|
| 1 | **Hồi quy tuyến tính đơn** | Y = aX + b (1 biến đầu vào) |
| 2 | **Hồi quy tuyến tính bội** | Y = a₁X₁ + a₂X₂ + ... + b (nhiều biến) |
| 3 | **Hồi quy logistic** | Phân loại 0/1, dùng hàm Sigmoid |
| 4 | **One-Hot Encoding** | `pd.get_dummies()` chuyển chữ → số |
| 5 | **Train/Test Split** | Luôn chia dữ liệu trước khi huấn luyện |
| 6 | **R²** | Đo chất lượng hồi quy (gần 1 = tốt) |
| 7 | **Accuracy / F1** | Đo chất lượng phân loại |

### Quy trình xây dựng mô hình:

```
Dữ liệu → Tiền xử lý → Chia Train/Test → .fit() → .predict() → Đánh giá
```

### So sánh với các tuần trước:

| Tuần | Nội dung | Vai trò trong tuần 16 |
|---|---|---|
| Week 7 (Pandas) | Đọc, xử lý dữ liệu | Chuẩn bị dữ liệu đầu vào |
| Week 8 (Visualization) | Vẽ biểu đồ | Trực quan mối quan hệ X-Y |
| Week 9 (Thống kê) | Thống kê mô tả | Hiểu phân bố dữ liệu |
| Week 14 (Tương quan) | Hệ số tương quan | Tương quan mạnh → hồi quy tốt |
| **Week 16** | **Hồi quy** | **Bước đầu vào Machine Learning!** |

In [ ]:
# Chúc mừng bạn đã hoàn thành bài học!
# Hãy thử thay đổi các con số trong code và chạy lại để hiểu sâu hơn.

print('=' * 50)
print('  HOÀN THÀNH BÀI HỌC TUẦN 16: HỒI QUY')
print('=' * 50)
print()
print('  Bạn đã học được:')
print('  ✓ Hồi quy tuyến tính đơn')
print('  ✓ Hồi quy tuyến tính bội + One-Hot Encoding')
print('  ✓ Hồi quy Logistic (phân loại)')
print('  ✓ Đánh giá mô hình (R², MSE, Accuracy, F1)')
print()
print('  Có câu hỏi? Hãy hỏi bất cứ lúc nào!')